<a href="https://colab.research.google.com/github/tseng097/Circuits-Computation-Biology/blob/main/EE5393_hw2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem1

Fibonacci iteration using molecular concentrations of species. We set 2 init values and take the iteration. At iteration k, we have two moleculars $A_k,B_k$. Hence, the next iteration is:
$$(A_{k+1},B_{k+1}) = (B_{k},A_{k}+B_{k})$$
The final output is $S_{12}$


In detail:
$S_k$: stage molecular
$T_k$: temporary molecular
$$
\text{For each } k=0,1,\dots,11:
$$

$$
S_k + B_k \xrightarrow{k_s} S_k + A_{k+1}
$$

$$
A_{k+1}=B_k
$$

$$
S_k + A_k \xrightarrow{k_s} S_k + T_k
$$

$$
S_k + B_k \xrightarrow{k_s} S_k + T_k
$$

$$
[T_k]=[A_k]+[B_k]
$$

$$
S_k + T_k \xrightarrow{k_s} S_k + B_{k+1}
$$

$$
[B_{k+1}] = [A_k]+[B_k]
$$

$$
S_k \longrightarrow S_{k+1}
$$

$$
(A_{k+1},B_{k+1})=(B_k,\;A_k+B_k)
$$

In [ ]:
def fibonacci_stage_style(a0: int, b0: int, steps: int = 12):
    A = [0] * (steps + 1)
    B = [0] * (steps + 1)

    A[0] = a0
    B[0] = b0

    for k in range(steps):
        # Sk + Bk -> Sk + A(k+1)
        A[k + 1] = B[k]
        # Sk + Ak -> Sk + B(k+1)
        # Sk + Bk -> Sk + B(k+1)
        B[k + 1] = A[k] + B[k]

    return A, B


A, B = fibonacci_stage_style(0, 1, 12)
print("For (0,1):")
for k in range(13):
    print(f"step {k:2d}: A{k}={A[k]}, B{k}={B[k]}")
print("Output =", A[12])   # 144

print("=================")

A, B = fibonacci_stage_style(3, 7, 12)
print("For (3,7):")
for k in range(13):
    print(f"step {k:2d}: A{k}={A[k]}, B{k}={B[k]}")
print("Output =", A[12])   # 1275

For (0,1):
step  0: A0=0, B0=1
step  1: A1=1, B1=1
step  2: A2=1, B2=2
step  3: A3=2, B3=3
step  4: A4=3, B4=5
step  5: A5=5, B5=8
step  6: A6=8, B6=13
step  7: A7=13, B7=21
step  8: A8=21, B8=34
step  9: A9=34, B9=55
step 10: A10=55, B10=89
step 11: A11=89, B11=144
step 12: A12=144, B12=233
Output = 144
For (3,7):
step  0: A0=3, B0=7
step  1: A1=7, B1=10
step  2: A2=10, B2=17
step  3: A3=17, B3=27
step  4: A4=27, B4=44
step  5: A5=44, B5=71
step  6: A6=71, B6=115
step  7: A7=115, B7=186
step  8: A8=186, B8=301
step  9: A9=301, B9=487
step 10: A10=487, B10=788
step 11: A11=788, B11=1275
step 12: A12=1275, B12=2063
Output = 1275


## Problem2

In [ ]:
def simulate_biquad_crn(inputs):
    # State variables representing molecular concentrations in the delay lines
    R1, G1, B1 = 0.0, 0.0, 0.0
    R2, G2, B2 = 0.0, 0.0, 0.0

    print(f"{'Cycle':<7} | {'Input (X)':<10} | {'Output (Y)':<10}")
    print("-" * 35)

    for cycle, X in enumerate(inputs, 1):
        # ---------------------------------------------------------
        # Phase 1: Green to Blue (Clocked by 'r')
        # ---------------------------------------------------------
        B1 += G1; G1 = 0.0
        B2 += G2; G2 = 0.0

        # ---------------------------------------------------------
        # Phase 2: Blue to Red & Computation (Clocked by 'g')
        # ---------------------------------------------------------
        Y = 0.0  # Output is consumed by an external sink each cycle

        # Fan-out operations (1 molecule becomes multiple distinct purpose molecules)
        B1_A, B1_Y, B1_R = B1, B1, B1
        B2_A, B2_Y = B2, B2
        B1, B2 = 0.0, 0.0  # Reactants consumed

        # Accumulate node A (Input + Feedback)
        # 8 molecules of B yield 1 molecule of A (1/8 scalar multiplication)
        A = X + (B1_A / 8.0) + (B2_A / 8.0)

        # Fan-out Node A
        A_Y, A_R = A, A

        # Accumulate Output Y (Feedforward sum)
        Y += (A_Y / 8.0) + (B1_Y / 8.0) + (B2_Y / 8.0)

        # Transfer states to Red delay elements
        R1 += A_R
        R2 += B1_R

        # ---------------------------------------------------------
        # Phase 3: Red to Green (Clocked by 'b')
        # ---------------------------------------------------------
        G1 += R1; R1 = 0.0
        G2 += R2; R2 = 0.0

        print(f"{cycle:<7} | {X:<10} | {Y:.4f}")

# Execute the simulation with the given input values
X_inputs = [100, 5, 500, 20, 250]
simulate_biquad_crn(X_inputs)

Cycle   | Input (X)  | Output (Y)
-----------------------------------
1       | 100        | 12.5000
2       | 5          | 14.6875
3       | 500        | 79.0234
4       | 20         | 77.3389
5       | 250        | 115.7953
